# Columnas Multicondicionales: Flexibilidad con `np.select()`

## 🎯 Objetivos
- Implementar la creación de columnas con más de dos categorías posibles.
- Comprender la estructura de `np.select()` (condiciones vs. elecciones).
- Diferenciar cuándo utilizar `np.where()` frente a `np.select()`.

## 📖 Introducción

Cuando necesitamos clasificar datos en más de dos categorías, el uso de `np.where()` se vuelve ineficiente porque nos obligaría a anidar múltiples funciones (un `np.where` dentro de otro), creando un código difícil de leer y mantener.

Para resolver esto, NumPy nos ofrece `np.select()`, que funciona como un bloque `if-elif-elif-else` optimizado para arrays. Nos permite definir una lista de condiciones y una lista correspondiente de valores, asignando el primer valor cuya condición sea verdadera.

## 🌉 Puente Pedagógico: La Estructura de Mapeo

En lugar de pensar en una decisión secuencial, piensa en `np.select()` como una **tabla de mapeo**. 

1. Defines una lista de **Condiciones** (el "¿Qué busco?").
2. Defines una lista de **Elecciones** (el "¿Qué asigno?").
3. Pandas recorre las condiciones en orden y asigna el primer valor que coincida.

### Diagrama de Flujo de `np.select()`
```
[ Lista de Condiciones ]         [ Lista de Valores ]
  Condición 1  -------------->   Elección 1 (Si C1 es True)
  Condición 2  -------------->   Elección 2 (Si C2 es True)
  Condición 3  -------------->   Elección 3 (Si C3 es True)
       ...
  Valor por Defecto <----------  Si NINGUNA condición es True
```

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path('laptop_price.csv')
df_laptops = pd.read_csv(DATA_PATH)
df_laptops.head()

## 🛠️ Implementación

### 1. Segmentación de Precios en 4 Niveles

Vamos a crear una columna `Price_tier` con la siguiente lógica:
- Precio > 3000 $\rightarrow$ `'Too Expensive'`
- Precio > 2000 $\rightarrow$ `'Expensive'`
- Precio > 800 $\rightarrow$ `'Affordable'`
- Precio $\le$ 800 $\rightarrow$ `'Cheap'`

In [ ]:
# 1. Definir la lista de condiciones (en orden de prioridad)
conditions = [
    (df_laptops['Price_euros'] > 3000),
    (df_laptops['Price_euros'] > 2000),
    (df_laptops['Price_euros'] > 800),
    (df_laptops['Price_euros'] <= 800)
]

# 2. Definir los valores correspondientes
choices = ['Too Expensive', 'Expensive', 'Affordable', 'Cheap']

# 3. Aplicar np.select
df_laptops['Price_tier'] = np.select(conditions, choices, default='Unknown')

df_laptops[['Price_euros', 'Price_tier']].head()

### 2. Verificación de Segmentos

Utilizamos `value_counts()` para asegurarnos de que la segmentación se haya realizado correctamente.

In [ ]:
counts = df_laptops['Price_tier'].value_counts()
print(f"Distribución de categorías de precio:\n{counts}")

## 📝 Ejercicios de Práctica

1. **Categorización de Pantalla**: Crea la columna `Screen_size_cat` usando `np.select()`:
   - `> 16` $\rightarrow$ `'Too Big'`
   - `> 14` $\rightarrow$ `'Big'`
   - `> 12` $\rightarrow$ `'Small'`
   - `\le 12` $\rightarrow$ `'Too Small'`
2. **Análisis de Memoria RAM**: 
   - Primero, crea una columna numérica `Ram_GB` extrayendo el número de la columna `Ram` (pista: `df['Ram'].str.replace('GB', '').astype(int)`).
   - Luego, usa `np.select` para categorizar: `\ge 32` $\rightarrow$ `'Ultra'`, `\ge 16` $\rightarrow$ `'High'`, `\ge 8` $\rightarrow$ `'Mid'`, resto $\rightarrow$ `'Low'`.
3. **Clasificación de Peso**: Crea una columna `Weight_cat` basada en la columna `Weight` (convirtiéndola a float eliminando 'kg').
   - `> 2.5` $\rightarrow$ `'Heavy'`, `> 1.5` $\rightarrow$ `'Medium'`, resto $\rightarrow$ `'Light'`.

In [ ]:
# Ejercicio 1

# Ejercicio 2

# Ejercicio 3


## 📋 Resumen Rápido: `np.where` vs `np.select`

| Característica | `np.where()` | `np.select()` |
| :--- | :--- | :--- |
| **Nº de Opciones** | Binario (2 opciones) | Múltiple (N opciones) |
| **Complejidad** | Baja / Simple | Media / Estructurada |
| **Sintaxis** | `(cond, T, F)` | `([conds], [vals], default)` |
| **Uso ideal** | Flags, Sí/No, Alto/Bajo | Segmentación, Niveles, Rangos |